In [17]:
"""
US Weather Heatmap vs Grain & Oilseed Futures Prices
======================================================

Builds a same-day comparison between:
1. Current weather (temp + precip) across major US grain-growing states
2. Today's price/% change in grain & oilseed futures

Run on your own machine (needs internet access to Open-Meteo + Yahoo Finance).

SETUP:
    pip install requests pandas plotly yfinance

Then run:
    python us_weather_vs_grain_prices.py
"""

import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import yfinance as yf


# ---------------------------------------------------------------------------
# 1. GROWING REGIONS -- representative lat/lon per major producing state
# ---------------------------------------------------------------------------

GROWING_REGIONS = {
    # State           (lat,     lon,      primary crops)
    "Iowa":           (41.878, -93.098,  "Corn / Soybeans"),
    "Illinois":       (40.349, -88.986,  "Corn / Soybeans"),
    "Nebraska":       (41.493, -99.901,  "Corn / Soybeans"),
    "Minnesota":      (45.694, -93.900,  "Corn / Soybeans"),
    "Indiana":        (40.267, -86.135,  "Corn / Soybeans"),
    "Kansas":         (38.500, -98.000,  "Wheat"),
    "South Dakota":   (44.500, -100.226, "Corn / Soybeans / Wheat"),
    "Ohio":           (40.417, -82.907,  "Corn / Soybeans"),
    "Missouri":       (38.457, -92.189,  "Corn / Soybeans"),
    "North Dakota":   (47.551, -101.002, "Wheat / Soybeans"),
    "Texas":          (31.968, -99.902,  "Wheat / Cotton"),
    "Oklahoma":       (35.467, -97.516,  "Wheat"),
}


# ---------------------------------------------------------------------------
# 2. FETCH CURRENT WEATHER PER REGION
# ---------------------------------------------------------------------------

def fetch_current_weather(lat, lon):
    """
    Pulls today's max temperature (C) and precipitation sum (mm) using
    Open-Meteo -- free, no API key required.
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": "temperature_2m_max,precipitation_sum",
        "forecast_days": 1,
        "timezone": "auto",
    }
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    daily = resp.json()["daily"]
    return {
        "temp_max_c": daily["temperature_2m_max"][0],
        "precip_mm": daily["precipitation_sum"][0],
    }


def build_weather_dataframe():
    rows = []
    for state, (lat, lon, crops) in GROWING_REGIONS.items():
        try:
            w = fetch_current_weather(lat, lon)
            rows.append({
                "state": state,
                "lat": lat,
                "lon": lon,
                "crops": crops,
                "temp_max_c": w["temp_max_c"],
                "temp_max_f": w["temp_max_c"] * 9 / 5 + 32,
                "precip_mm": w["precip_mm"],
            })
        except Exception as e:
            print(f"Weather fetch failed for {state}: {e}")
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# 3. FETCH TODAY'S GRAIN / OILSEED FUTURES PRICES
# ---------------------------------------------------------------------------

GRAIN_OILSEED_TICKERS = {
    "ZC=F": "Corn",
    "ZS=F": "Soybeans",
    "ZW=F": "Wheat",
    "ZL=F": "Soybean Oil",
    "ZM=F": "Soybean Meal",
}


def fetch_futures_snapshot():
    rows = []
    for ticker, name in GRAIN_OILSEED_TICKERS.items():
        t = yf.Ticker(ticker)
        info = t.fast_info
        last_price = info.get("lastPrice")
        prev_close = info.get("previousClose")
        change = (last_price - prev_close) if last_price and prev_close else None
        change_pct = (change / prev_close * 100) if change and prev_close else None
        rows.append({
            "Symbol": ticker,
            "Name": name,
            "Price": last_price,
            "Change": change,
            "Change %": change_pct,
        })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# 4. VISUALIZE -- weather heatmap + price bar chart side by side
# ---------------------------------------------------------------------------

def plot_weather_heatmap(weather_df, metric="temp_max_f"):
    """
    metric: 'temp_max_f' for heat map, or 'precip_mm' for rainfall map
    """
    color_scale = "YlOrRd" if metric == "temp_max_f" else "Blues"
    label = "Max Temp (F)" if metric == "temp_max_f" else "Precipitation (mm)"

    fig = px.scatter_geo(
        weather_df,
        lat="lat",
        lon="lon",
        color=metric,
        size=[20] * len(weather_df),
        hover_name="state",
        hover_data={"crops": True, "temp_max_f": True, "precip_mm": True,
                    "lat": False, "lon": False},
        color_continuous_scale=color_scale,
        scope="usa",
        title=f"US Grain Belt Weather Today -- {label}",
    )
    fig.update_geos(showland=True, landcolor="rgb(240,240,240)")
    return fig


def plot_price_changes(futures_df):
    colors = ["green" if x >= 0 else "red" for x in futures_df["Change %"]]
    fig = go.Figure(go.Bar(
        x=futures_df["Name"],
        y=futures_df["Change %"],
        marker_color=colors,
        text=futures_df["Change %"].round(2),
        textposition="outside",
    ))
    fig.update_layout(
        title="Today's Grain & Oilseed Futures % Change",
        yaxis_title="% Change",
        template="plotly_white",
    )
    return fig


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------

def main():
    print("Fetching weather data across grain belt states...")
    weather_df = build_weather_dataframe()
    print(weather_df)

    print("\nFetching today's grain & oilseed futures prices...")
    futures_df = fetch_futures_snapshot()
    print(futures_df)

    print("\nBuilding visualizations...")
    heat_fig = plot_weather_heatmap(weather_df, metric="temp_max_f")
    heat_fig.write_html("weather_heatmap_temp.html")
    heat_fig.show()

    rain_fig = plot_weather_heatmap(weather_df, metric="precip_mm")
    rain_fig.write_html("weather_heatmap_precip.html")
    rain_fig.show()

    price_fig = plot_price_changes(futures_df)
    price_fig.write_html("futures_price_changes.html")
    price_fig.show()

    # Simple flag: states that are both hot AND dry today
    weather_df["stress_flag"] = (weather_df["temp_max_f"] > 90) & (weather_df["precip_mm"] < 2)
    stressed_states = weather_df[weather_df["stress_flag"]]["state"].tolist()

    print(f"\nStates showing hot+dry stress today: {stressed_states}")
   
if __name__ == "__main__":
    main()

Fetching weather data across grain belt states...
           state     lat      lon                    crops  temp_max_c  \
0           Iowa  41.878  -93.098          Corn / Soybeans        28.8   
1       Illinois  40.349  -88.986          Corn / Soybeans        28.1   
2       Nebraska  41.493  -99.901          Corn / Soybeans        30.2   
3      Minnesota  45.694  -93.900          Corn / Soybeans        28.6   
4        Indiana  40.267  -86.135          Corn / Soybeans        28.1   
5         Kansas  38.500  -98.000                    Wheat        31.5   
6   South Dakota  44.500 -100.226  Corn / Soybeans / Wheat        30.8   
7           Ohio  40.417  -82.907          Corn / Soybeans        29.4   
8       Missouri  38.457  -92.189          Corn / Soybeans        29.9   
9   North Dakota  47.551 -101.002         Wheat / Soybeans        31.5   
10         Texas  31.968  -99.902           Wheat / Cotton        37.3   
11      Oklahoma  35.467  -97.516                    Wheat    


States showing hot+dry stress today: ['Texas']
